In [ ]:
from vpython import canvas, wtext, button, slider, sphere, vector, color, rate
import random

# 1. GLOBAL SYSTEM VARIABLES
is_running = True
grid_width = 9   
grid_height = 9  
spacing = 1.3 

# Filter Structure Matrix
grid_matrix = {}
filter_atoms = []

# Chemical Gas Configurations (Default: Air Purification)
gas_A = {"radius": 0.2, "speed": 3.0, "color": color.cyan, "name": "Nitrogen (N2)"}
gas_B = {"radius": 0.6, "speed": 1.5, "color": color.red,  "name": "Soot / Smoke"}
current_mode_name = "Air Purification (Smoke vs N2)"

# Action History Stacks (Undo / Redo)
undo_stack = []  
redo_stack = []  

# --- UI INTERACTIVE INTERACTION FUNCTIONS ---
def set_mode_air(b):
    global gas_A, gas_B, current_mode_name
    gas_A = {"radius": 0.2, "speed": 3.0, "color": color.cyan, "name": "Nitrogen (N2)"}
    gas_B = {"radius": 0.6, "speed": 1.5, "color": color.red,  "name": "Soot / Smoke"}
    current_mode_name = "Air Purification (Smoke vs N2)"; reset_statistics()

def set_mode_hydrogen(b):
    global gas_A, gas_B, current_mode_name
    gas_A = {"radius": 0.15, "speed": 4.0, "color": color.yellow, "name": "Hydrogen (H2)"}
    gas_B = {"radius": 0.40, "speed": 2.2, "color": color.orange, "name": "Methane (CH4)"}
    current_mode_name = "Hydrogen Extraction (H2 from CH4)"; reset_statistics()

def set_mode_isotopes(b):
    global gas_A, gas_B, current_mode_name
    gas_A = {"radius": 0.30, "speed": 2.5, "color": color.green, "name": "Light Isotope"}
    gas_B = {"radius": 0.36, "speed": 2.4, "color": color.magenta, "name": "Heavy Isotope"}
    current_mode_name = "Isotope Separation (High Precision)"; reset_statistics()

def reset_statistics():
    global blue_total, blue_passed, red_total, red_blocked, all_gas_molecules
    for gas in all_gas_molecules: gas.visible = False
    all_gas_molecules.clear()
    blue_total = blue_passed = red_total = red_blocked = 0

def toggle_simulation(b):
    global is_running; is_running = not is_running
    if is_running: b.text = "⏸️ PAUSE & EDIT STRUCTURE"
    else: b.text = "▶️ RUN MOLECULAR FLOW"

def click_action(evt):
    if not is_running:
        clicked_object = sim_canvas.mouse.pick
        if clicked_object and clicked_object in filter_atoms:
            coord = clicked_object.grid_coord
            undo_stack.append(coord)
            redo_stack.clear()
            clicked_object.visible = False
            filter_atoms.remove(clicked_object)
            if coord in grid_matrix:
                del grid_matrix[coord]

def undo_action(b):
    if not is_running and len(undo_stack) > 0:
        coord = undo_stack.pop()
        redo_stack.append(coord)
        y, z = coord
        half_w = grid_width / 2; half_h = grid_height / 2
        pos_vector = vector(0, (y - half_h + 0.5) * spacing, (z - half_w + 0.5) * spacing)
        restored_atom = sphere(canvas=sim_canvas, pos=pos_vector, radius=0.5, color=color.gray(0.6))
        restored_atom.grid_coord = coord
        filter_atoms.append(restored_atom)
        grid_matrix[coord] = restored_atom

def redo_action(b):
    if not is_running and len(redo_stack) > 0:
        coord = redo_stack.pop()
        undo_stack.append(coord)
        if coord in grid_matrix:
            atom_to_remove = grid_matrix[coord]
            atom_to_remove.visible = False
            filter_atoms.remove(atom_to_remove)
            del grid_matrix[coord]

def change_width(s): global grid_width; grid_width = int(s.value); apply_custom_size()
def change_height(s): global grid_height; grid_height = int(s.value); apply_custom_size()

def apply_custom_size():
    global blue_total, blue_passed, red_total, red_blocked, all_gas_molecules, half_w, half_h
    for atom in filter_atoms: atom.visible = False
    filter_atoms.clear()
    for gas in all_gas_molecules: gas.visible = False
    all_gas_molecules.clear()
    undo_stack.clear(); redo_stack.clear(); grid_matrix.clear()
    blue_total = blue_passed = red_total = red_blocked = 0
    half_w = grid_width / 2; half_h = grid_height / 2
    
    for y in range(0, grid_height):
        for z in range(0, grid_width):
            pos_vector = vector(0, (y - half_h + 0.5) * spacing, (z - half_w + 0.5) * spacing)
            atom = sphere(canvas=sim_canvas, pos=pos_vector, radius=0.5, color=color.gray(0.6))
            atom.grid_coord = (y, z) 
            filter_atoms.append(atom)
            grid_matrix[(y, z)] = atom

# 2. 3D CANVAS & INITIALIZATION
sim_canvas = canvas(title="<h3>MolFlow v1.0 — Global Web Release</h3>", 
                    width=700, height=380, center=vector(0, 0, 0), background=color.gray(0.15))
sim_canvas.bind('mousedown', click_action)

print("\n")
control_button = button(bind=toggle_simulation, text="⏸️ PAUSE & EDIT STRUCTURE")
print("   ")
undo_button = button(bind=undo_action, text="↩️ Undo")
print(" ")
redo_button = button(bind=redo_action, text="↪️ Redo")

print("\n\n📐 Membrane Geometry Adjustments:")
print("Width (Atoms): ")
width_slider = slider(min=1, max=20, value=9, step=1, bind=change_width)
print("\nHeight (Atoms): ")
height_slider = slider(min=1, max=20, value=9, step=1, bind=change_height)

print("\n\n🧪 Select Chemical Gas Mixture for Injection:")
btn_mode1 = button(bind=set_mode_air, text="💨 Air Purification (Smoke)")
print("  ")
btn_mode2 = button(bind=set_mode_hydrogen, text="🔋 Hydrogen Extraction (H2)")
print("  ")
btn_mode3 = button(bind=set_mode_isotopes, text="⚛️ Isotope Separation")

print("\n\n")
stats_panel = wtext(text="Initializing Interface...")

apply_custom_size()

blue_total = blue_passed = red_total = red_blocked = 0
all_gas_molecules = []
dt = 0.02
frame_counter = 0

# 3. MAIN SIMULATION LOOP
while True:
    rate(60)
    if is_running:
        frame_counter += 1
        if frame_counter % 15 == 0:
            spawn_limit_y = half_h * spacing
            spawn_limit_z = half_w * spacing
            start_pos = vector(-8, random.uniform(-spawn_limit_y, spawn_limit_y), random.uniform(-spawn_limit_z, spawn_limit_z))
            
            if random.random() > 0.5:
                new_gas = sphere(canvas=sim_canvas, pos=start_pos, radius=gas_A["radius"], color=gas_A["color"])
                new_gas.velocity = vector(random.uniform(gas_A["speed"]*0.8, gas_A["speed"]*1.2), random.uniform(-0.2, 0.2), random.uniform(-0.2, 0.2))
                new_gas.type = "blue"; blue_total += 1
            else:
                new_gas = sphere(canvas=sim_canvas, pos=start_pos, radius=gas_B["radius"], color=gas_B["color"])
                new_gas.velocity = vector(random.uniform(gas_B["speed"]*0.8, gas_B["speed"]*1.2), random.uniform(-0.1, 0.1), random.uniform(-0.1, 0.1))
                new_gas.type = "red"; red_total += 1
            all_gas_molecules.append(new_gas)
            
        for gas in all_gas_molecules[:]:
            gas.pos = gas.pos + gas.velocity * dt
            for atom in filter_atoms:
                distance = (gas.pos - atom.pos).mag
                min_dist = gas.radius + atom.radius
                if distance < min_dist:
                    normal = (gas.pos - atom.pos).norm()
                    gas.velocity = gas.velocity - 2 * gas.velocity.dot(normal) * normal
                    gas.pos = atom.pos + normal * min_dist
                    
            if gas.type == "blue" and gas.pos.x > 6:
                blue_passed += 1; gas.visible = False; all_gas_molecules.remove(gas)
            elif gas.type == "red" and gas.pos.x < -9:
                red_blocked += 1; gas.visible = False; all_gas_molecules.remove(gas)
            elif gas.pos.x > 8 or gas.pos.x < -10:
                gas.visible = False; all_gas_molecules.remove(gas)

    # 4. LUXURY DARK-THEME DASHBOARD (STYLING UPDATE)
    efficiency = (red_blocked / red_total * 100) if red_total > 0 else 0
    flow_rate = (blue_passed / blue_total * 100) if blue_total > 0 else 0
    
    status_text = "<span style='color: #2ecc71; font-weight: bold;'>FLOW ACTIVE</span>" if is_running else "<span style='color: #e67e22; font-weight: bold;'>STRUCTURE EDIT MODE</span>"
    
    # CSS-Styling applied inside the <div> block for a modern look
    stats_panel.text = (
        f"<div style='border: none; padding: 20px; background-color: #2c3e50; color: #ecf0f1; "
        f"font-family: \"Segoe UI\", sans-serif; width: 660px; border-radius: 10px; "
        f"box-shadow: 0 4px 15px rgba(0,0,0,0.3); margin-top: 15px;'>"
        f"<h3 style='margin-top: 0; color: #3498db; border-bottom: 2px solid #34495e; padding-bottom: 10px;'>📊 MolFlow Telemetry Core v1.0</h3>"
        f"<b>Current Chemical Mode:</b> <span style='color: #f1c40f; font-weight: bold;'>{current_mode_name}</span><br>"
        f"<b>System Status:</b> {status_text}<br>"
        f"<b>Active Membrane Geometry:</b> {grid_width}w x {grid_height}h atoms<br>"
        f"<div style='margin: 15px 0; border-top: 1px solid #34495e;'></div>"
        f"<b>BLOCKED {gas_B['name'].upper()}:</b> {red_blocked} / {red_total}<br>"
        f"🎯 <span style='color: #e74c3c; font-weight: bold; font-size: 16px;'>FILTRATION EFFICIENCY: {efficiency:.1f}%</span><br>"
        f"<div style='margin: 10px 0;'></div>"
        f"<b>PASSED {gas_A['name'].upper()}:</b> {blue_passed} / {blue_total}<br>"
        f"⚡ <span style='color: #1abc9c; font-weight: bold; font-size: 16px;'>THROUGHPUT CAPACITY: {flow_rate:.1f}%</span>"
        f"</div>"
    )